# Import library

In [1]:
import pandas as pd
from pathlib import Path

# Menentukan folder sumber data

In [2]:
input_dir = Path.cwd().parent / "data" / "extracted"

# Membaca seluruh file Parquet

In [3]:
data = {}

for file in input_dir.glob("*.parquet"):
    table_name = file.stem

    data[table_name] = pd.read_parquet(file)

    print(f"{table_name}: {data[table_name].shape}")

stg_customers: (25000, 18)
stg_customer_reviews: (77530, 8)
stg_marketing_campaigns: (151, 14)
stg_orders: (100000, 16)
stg_order_items: (254331, 9)
stg_payments: (100000, 8)
stg_products: (550, 14)
stg_returns: (12075, 8)
stg_shipments: (89681, 11)


# Memastikan data berhasil dibaca

In [4]:
data["stg_customers"].head()

,customer_id,customer_signup_date,gender,age,age_group,state,city,pincode_prefix,customer_segment,preferred_device,preferred_payment_method,acquisition_channel,total_orders,total_spend,average_order_value,last_order_date,customer_status,loyalty_tier
0,CUST100000,2023-01-09,Male,44.0,35-44,Madhya Pradesh,Indore,607,Regular,Mobile App,EMI,Affiliate,3,36884.57,12294.86,2023-05-26,Churned,Bronze
1,CUST100001,2023-04-10,Male,20.0,18-24,West Bengal,Kolkata,245,Regular,Mobile App,Cash on Delivery,Organic Search,7,371585.73,53083.68,2024-05-23,Churned,Gold
2,CUST100002,2025-05-09,Female,35.0,35-44,Gujarat,Ahmedabad,160,Premium,Mobile Web,UPI,SMS Marketing,10,114901.74,11490.17,2025-07-14,At Risk,Silver
3,CUST100003,2024-12-24,Female,34.0,25-34,Maharashtra,Pune,175,New,Mobile App,UPI,Influencer Marketing,1,3788.82,3788.82,2025-12-19,Active,Bronze
4,CUST100004,2023-08-15,Male,38.0,35-44,Andhra Pradesh,Visakhapatnam,446,Budget,Desktop,Cash on Delivery,Referral,2,61295.90,30647.95,2023-11-01,Churned,Silver


# Data profiling

In [14]:
# 1. Profiling per kolom
def profiling_table(table_name):
    df = data[table_name]

    profile = pd.DataFrame([
        {
            'column': column,
            'dtype': str(df[column].dtype),
            'missing': int(df[column].isna().sum()),
            'missing_pct': round(df[column].isna().mean() * 100, 2),
            'unique': int(df[column].nunique(dropna=True)),
        }
        for column in df.columns
    ]).sort_values(['missing_pct', 'column'], ascending=[False, True]).reset_index(drop=True)
    print('\nProfiling per kolom')
    display(profile)

# 2. Statistik deskriptif kolom numerik
def statistics_table(table_name):
    df = data[table_name]

    numeric_columns = df.select_dtypes(include='number').columns
    print('Statistik deskriptif')
    if len(numeric_columns) > 0:
        display(df[numeric_columns].describe().T)
    else:
        print('Tidak ada kolom numerik.')

# 3. Lima nilai paling sering pada kolom kategorikal
def categorical_analysis(table_name):
    df = data[table_name]
    categorical_columns = df.select_dtypes(include=['object', 'category', 'bool']).columns
    categorical_rows = []
    for column in categorical_columns:
        for value, count in df[column].value_counts(dropna=False).head(5).items():
            categorical_rows.append({'column': column, 'value': value, 'count': int(count)})

    print('Lima nilai paling sering untuk kolom kategorikal')
    if categorical_rows:
        display(pd.DataFrame(categorical_rows))
    else:
        print('Tidak ada kolom kategorikal.')

## Customers

In [15]:
profiling_table("stg_customers")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,last_order_date,object,2300,9.2,1074
1,age,float64,250,1.0,49
2,preferred_device,object,125,0.5,3
3,acquisition_channel,object,0,0.0,10
4,age_group,object,0,0.0,5
5,average_order_value,float64,0,0.0,22373
6,city,object,0,0.0,31
7,customer_id,object,0,0.0,25000
8,customer_segment,object,0,0.0,4
9,customer_signup_date,object,0,0.0,1096


In [16]:
statistics_table("stg_customers")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
age,24750.0,31.819232,8.520634,18.0,25.000,32.000,38.0000,67.00
pincode_prefix,25000.0,499.503960,231.231990,100.0,298.000,501.000,700.0000,899.00
total_orders,25000.0,3.884800,3.317464,0.0,1.000,3.000,5.0000,24.00
total_spend,25000.0,100031.110244,130582.708641,0.0,11598.905,47466.040,140534.1050,1310772.39
average_order_value,25000.0,22135.360868,26901.858104,0.0,5633.305,14197.185,29427.0375,381793.12


In [17]:
categorical_analysis("stg_customers")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,customer_id,CUST124999,1
1,customer_id,CUST100000,1
2,customer_id,CUST100001,1
3,customer_id,CUST100002,1
4,customer_id,CUST100003,1
5,customer_signup_date,2023-02-28,68
6,customer_signup_date,2023-01-15,60
7,customer_signup_date,2023-03-10,59
8,customer_signup_date,2023-01-02,59
9,customer_signup_date,2023-04-05,58


## Product

In [18]:
profiling_table("stg_products")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,rating_average,float64,4,0.73,23
1,brand,object,0,0.00,148
2,category,object,0,0.00,22
3,cost_price,float64,0,0.00,549
4,discount_range,object,0,0.00,7
5,price,float64,0,0.00,316
6,product_id,object,0,0.00,550
7,product_launch_date,object,0,0.00,439
8,product_name,object,0,0.00,549
9,product_type,object,0,0.00,4


In [19]:
statistics_table("stg_products")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
price,550.0,4429.636364,10207.044348,50.0000,450.00000,1075.0000,2962.500000,76020.0000
cost_price,550.0,3531.198109,8786.803705,34.1200,307.36000,717.1850,2076.402500,68974.9600
rating_average,546.0,4.006227,0.430870,2.8000,3.70000,4.0000,4.300000,5.0000
rating_count,550.0,126.940000,237.043143,0.0000,22.00000,54.0000,131.750000,2161.0000
stock_quantity,550.0,1487.603636,859.794366,11.0000,759.50000,1521.0000,2215.250000,2994.0000
return_rate_baseline,550.0,0.075462,0.052584,0.0059,0.03325,0.0665,0.102325,0.2679


In [20]:
categorical_analysis("stg_products")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,product_id,PROD10549,1
1,product_id,PROD10000,1
2,product_id,PROD10001,1
3,product_id,PROD10002,1
4,product_id,PROD10003,1
5,product_name,Morar Basic Art Supplie,2
6,product_name,"Sami, Premium Home Decor",1
7,product_name,Behl-Chanda Classic Medical Device,1
8,product_name,Goda Ultra Office Furniture,1
9,product_name,Madan-Kanda Ultra Travel Bag,1


## Order

In [21]:
profiling_table("stg_orders")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,campaign_id,object,82821,82.82,151
1,coupon_code,object,33711,33.71,8
2,customer_id,object,0,0.00,22820
3,delivery_city,object,0,0.00,31
4,delivery_state,object,0,0.00,15
5,discount_percentage,float64,0,0.00,7
6,final_amount,float64,0,0.00,70941
7,marketing_channel,object,0,0.00,10
8,order_date,object,0,0.00,1096
9,order_id,object,0,0.00,100000


In [22]:
statistics_table("stg_orders")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
order_time,100000,0 days 15:29:09.631460,0 days 05:15:05.388272370,0 days 00:00:02,0 days 11:46:45.750000,0 days 16:25:06,0 days 19:47:26,0 days 23:59:59
discount_percentage,100000.0,11.98425,10.687991,0.0,0.0,10.0,20.0,30.0
subtotal,100000.0,24498.989743,42814.400978,33.5,3449.6375,9050.775,24531.3,578151.0
shipping_fee,100000.0,1.98408,12.409613,0.0,0.0,0.0,0.0,149.0
tax_amount,100000.0,1224.949862,2140.720064,1.68,172.48,452.535,1226.56,28907.55
final_amount,100000.0,25725.923686,44954.001129,71.0,3622.1175,9503.31,25757.86,607058.55


In [23]:
categorical_analysis("stg_orders")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,order_id,ORD1099960,1
1,order_id,ORD1099961,1
2,order_id,ORD1099962,1
3,order_id,ORD1099963,1
4,order_id,ORD1099964,1
5,customer_id,CUST107174,24
6,customer_id,CUST122116,23
7,customer_id,CUST114755,23
8,customer_id,CUST111225,23
9,customer_id,CUST104958,22


## Order Item

In [24]:
profiling_table("stg_order_items")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,discount_percentage,float64,0,0.0,21
1,item_cost,float64,0,0.0,3408
2,item_revenue,float64,0,0.0,16441
3,order_id,object,0,0.0,100000
4,order_item_id,object,0,0.0,254331
5,product_id,object,0,0.0,550
6,profit,float64,0,0.0,33810
7,quantity,int64,0,0.0,8
8,unit_price,float64,0,0.0,316


In [25]:
statistics_table("stg_order_items")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
quantity,254331.0,2.302669,1.138847,1.00,1.00,2.00,3.00,8.00
unit_price,254331.0,4796.386992,11229.239979,50.00,570.00,1210.00,3150.00,76020.00
discount_percentage,254331.0,12.899855,10.741200,0.00,3.00,11.50,21.50,33.00
item_revenue,254331.0,9632.718679,25845.808896,33.50,884.00,2245.80,6289.50,577752.00
item_cost,254331.0,8856.980185,25305.466331,34.12,713.76,1772.40,5141.92,551799.68
profit,254331.0,775.738494,3639.700982,-108249.36,63.06,315.29,982.41,75057.36


In [26]:
categorical_analysis("stg_order_items")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,order_item_id,OI10254330,1
1,order_item_id,OI10000000,1
2,order_item_id,OI10000001,1
3,order_item_id,OI10000002,1
4,order_item_id,OI10000003,1
5,order_id,ORD1046330,5
6,order_id,ORD1099992,5
7,order_id,ORD1078519,5
8,order_id,ORD1099945,5
9,order_id,ORD1046328,5


## Payment

In [27]:
profiling_table("stg_payments")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,amount_paid,float64,0,0.0,66618
1,order_id,object,0,0.0,100000
2,payment_date,object,0,0.0,1095
3,payment_id,object,0,0.0,100000
4,payment_method,object,0,0.0,10
5,payment_status,object,0,0.0,5
6,refund_amount,float64,0,0.0,9696
7,transaction_fee,float64,0,0.0,17575


In [28]:
statistics_table("stg_payments")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
amount_paid,100000.0,23910.496199,43921.744572,0.0,2714.9425,8278.78,23473.8000,607058.55
transaction_fee,100000.0,79.654963,331.410497,0.0,0.0000,0.00,2.4525,7936.84
refund_amount,100000.0,2687.408116,16340.816610,0.0,0.0000,0.00,0.0000,541686.50


In [29]:
categorical_analysis("stg_payments")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,payment_id,PAY1099960,1
1,payment_id,PAY1099961,1
2,payment_id,PAY1099962,1
3,payment_id,PAY1099963,1
4,payment_id,PAY1099964,1
5,order_id,ORD1099960,1
6,order_id,ORD1099961,1
7,order_id,ORD1099962,1
8,order_id,ORD1099963,1
9,order_id,ORD1099964,1


## Shipment

In [30]:
profiling_table("stg_shipments")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,actual_delivery_date,object,2321,2.59,1091
1,delivery_days,float64,2321,2.59,15
2,delayed_flag,int64,0,0.00,2
3,delivery_city,object,0,0.00,31
4,delivery_status,object,0,0.00,3
5,dispatch_date,object,0,0.00,1095
6,expected_delivery_date,object,0,0.00,1091
7,order_id,object,0,0.00,89681
8,shipment_id,object,0,0.00,89681
9,shipping_method,object,0,0.00,5


In [31]:
statistics_table("stg_shipments")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
delivery_days,87360.0,4.102564,2.507985,0.0,2.0,4.0,6.0,14.0
delayed_flag,89681.0,0.417569,0.493161,0.0,0.0,0.0,1.0,1.0


In [32]:
categorical_analysis("stg_shipments")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,shipment_id,SHIP1089680,1
1,shipment_id,SHIP1000000,1
2,shipment_id,SHIP1000001,1
3,shipment_id,SHIP1000002,1
4,shipment_id,SHIP1000003,1
5,order_id,ORD1099999,1
6,order_id,ORD1000001,1
7,order_id,ORD1000002,1
8,order_id,ORD1000005,1
9,order_id,ORD1000006,1


## Return

In [33]:
profiling_table("stg_returns")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,customer_id,object,0,0.0,7285
1,order_id,object,0,0.0,9510
2,product_id,object,0,0.0,523
3,refund_amount,float64,0,0.0,4801
4,return_date,object,0,0.0,1078
5,return_id,object,0,0.0,12075
6,return_reason,object,0,0.0,8
7,return_status,object,0,0.0,4


In [34]:
statistics_table("stg_returns")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
refund_amount,12075.0,8770.052323,24672.226586,0.0,552.0,1960.15,5880.0,384411.0


In [35]:
categorical_analysis("stg_returns")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,return_id,RET112074,1
1,return_id,RET100000,1
2,return_id,RET100001,1
3,return_id,RET100002,1
4,return_id,RET100003,1
5,order_id,ORD1055196,5
6,order_id,ORD1002445,5
7,order_id,ORD1061471,5
8,order_id,ORD1065769,5
9,order_id,ORD1033171,5


## Review

In [36]:
profiling_table("stg_customer_reviews")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,review_sentiment,object,233,0.3,3
1,customer_id,object,0,0.0,19242
2,order_id,object,0,0.0,54363
3,product_id,object,0,0.0,550
4,rating,int64,0,0.0,5
5,review_date,object,0,0.0,1087
6,review_id,object,0,0.0,77530
7,verified_purchase,int64,0,0.0,2


In [37]:
statistics_table("stg_customer_reviews")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
rating,77530.0,3.691939,0.904587,1.0,3.0,4.0,4.0,5.0
verified_purchase,77530.0,0.971185,0.167286,0.0,1.0,1.0,1.0,1.0


In [38]:
categorical_analysis("stg_customer_reviews")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,review_id,REV177529,1
1,review_id,REV100000,1
2,review_id,REV100001,1
3,review_id,REV100002,1
4,review_id,REV100003,1
5,order_id,ORD1095972,5
6,order_id,ORD1092953,5
7,order_id,ORD1006391,5
8,order_id,ORD1013420,5
9,order_id,ORD1067884,5


## Campaign

In [40]:
profiling_table("stg_marketing_campaigns")


Profiling per kolom


,column,dtype,missing,missing_pct,unique
0,campaign_cost,float64,0,0.0,151
1,campaign_end_date,object,0,0.0,138
2,campaign_id,object,0,0.0,151
3,campaign_name,object,0,0.0,151
4,campaign_start_date,object,0,0.0,130
5,campaign_type,object,0,0.0,8
6,channel,object,0,0.0,10
7,clicks,int64,0,0.0,146
8,conversions,int64,0,0.0,110
9,discount_percentage,float64,0,0.0,6


In [41]:
statistics_table("stg_marketing_campaigns")

Statistik deskriptif


,count,mean,std,min,25%,50%,75%,max
discount_percentage,151.0,1.834437e+01,8.159286e+00,5.00,10.000,20.000,25.000,3.000000e+01
impressions,151.0,6.104181e+04,9.501690e+04,296.00,4133.000,15195.000,79192.500,6.426310e+05
clicks,151.0,1.949464e+03,1.763978e+03,30.00,686.000,1350.000,2579.000,1.179100e+04
conversions,151.0,1.137682e+02,8.262295e+01,2.00,62.500,90.000,141.000,7.000000e+02
campaign_cost,151.0,1.239211e+06,1.647686e+06,13777.19,503025.795,762609.670,1227259.100,1.275682e+07
revenue_generated,151.0,2.889379e+06,2.157381e+06,11607.02,1581971.355,2282681.180,3597974.345,1.925469e+07
roi,151.0,2.406000e+00,1.862961e+00,-0.60,1.046,1.935,3.696,7.639000e+00


In [42]:
categorical_analysis("stg_marketing_campaigns")

Lima nilai paling sering untuk kolom kategorikal


,column,value,count
0,campaign_id,CAMP1001,1
1,campaign_id,CAMP1002,1
2,campaign_id,CAMP1003,1
3,campaign_id,CAMP1004,1
4,campaign_id,CAMP1005,1
5,campaign_name,Direct/App Festival Special Jan 2023,1
6,campaign_name,Social Media - Instagram Flash Sale Jan 2023,1
7,campaign_name,Organic Search New Product Launch Jan 2023,1
8,campaign_name,SMS Marketing Festival Special Jan 2023,1
9,campaign_name,Email Marketing New Product Launch Feb 2023,1
